# 21 cm signal loss under foreground-mode filtering

The eigenmode analysis shows that the simulated beam-weighted foregrounds occupy a low-dimensional spectral subspace. On its own that says nothing about whether the cosmological signal survives the same filter, so here we push an ensemble of global 21 cm models through the *identical* projection onto the leading $N$ foreground modes and read the retained signal off the same axes as the residuals.

Panels (a1)/(a2) show one model per outcome class -- all three drawn from a common depth window (80-160 mK) so that trough *width*, not amplitude, is the visible difference -- before and after filtering $N$ modes. Each is highlighted as a thick curve of the same colour in panel (b), so the two sides can be read against each other. What survives is small but still structured -- note it is band-edge ringing from projecting onto a truncated smooth basis, not a residual trough, so retained RMS is not retained signal *shape*. Panel (b) puts the foreground residual, the worst-case $+1$ m position-error systematic, and 500 individual retained-signal curves coloured by outcome class on one set of axes. It supersedes `foreground_svd_residual.pdf` -- the black curve is the same one, now never shown without the signal beside it.

Models are classed by the RMS they retain at $N = 10$: below 1 mK, 1-5 mK, and above 5 mK, which splits the ensemble roughly into thirds. Higher cuts are not useful here -- 10 mK catches 2.6% of models and 25 mK none, since the most foreground-orthogonal model retains 16.4 mK. **The discriminator is trough width.** At matched depth the median width runs 73 / 30 / 17 MHz from the destroyed class to the surviving one: the smooth low-order foreground modes absorb broad troughs and leave narrow ones. Amplitude matters too, but most of the destroyed class is simply faint to begin with (median depth 2.7 mK).

$N = 10$ is the smallest $N$ at which *both* floors fall below the median retained signal. At $N = 8$ the foreground residual and the median signal are the same size.

**Limitations, to be stated wherever this result is used.** The modes come from a single simulated sky (GSM16) and beam, with no noise and no receiver systematics; in practice the basis would be estimated from data that already contain the signal, which costs additional signal loss not captured here. Filtering is a hard projection, whereas a joint signal-plus-foreground fit would recover some of what is removed. Signal loss is severe in absolute terms, and whether the retained amplitude is detectable is set by thermal noise and integration time, which this calculation does not model. This is a statement about spectral subspace overlap, not a sensitivity forecast.

**TODO:** confirm the provenance of the 21 cm model ensemble before writing the caption citation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
d = np.load("signal_loss.npz", allow_pickle=True)
freqs = d["freqs_MHz"]           # (n_f,) MHz
Vh = d["Vh"]                     # (n_f, n_f) foreground spectral modes
s_fg = d["s_fg"]                 # (n_f,) singular values of the T_ant waterfall
n_time = int(d["n_time"])        # LST samples in that waterfall
dT = d["dT_spectra"]             # (3, n_lst, n_f) +1 m E/N/U systematic [K]
T21 = d["T21_models"]            # (n_model, n_f) global-signal ensemble [K]
show_idx = d["show_idx"]         # the three models drawn in panels (a1)/(a2)
cls = d["cls"]                   # (n_model,) class 0/1/2 by retained RMS at N_ANCHOR
show_w = d["show_width_MHz"]     # trough width [MHz] of each drawn model
class_labels = [str(x) for x in d["class_labels"]]
n_f = freqs.size
N_SHOW = 18                      # x-axis extent
N_ANCHOR = int(d["n_anchor"])    # modes filtered at the quoted operating point
N_CURVES = 500                   # individual signals drawn in panel (b)
CURVE_ALPHA = 0.10               # opacity of those curves
print(f"{T21.shape[0]} 21 cm models on {n_f} channels, "
      f"{freqs[0]:.0f}-{freqs[-1]:.0f} MHz")

In [ ]:
n_modes = np.arange(N_SHOW + 1)

# Foreground residual after filtering the leading N modes [K] -- the curve
# that used to be foreground_svd_residual.pdf.
tail = np.concatenate([np.cumsum(s_fg[::-1] ** 2)[::-1], [0.0]])
fg_resid = np.sqrt(tail / (n_time * n_f))[: N_SHOW + 1]


def filt_rms(x):
    """RMS over frequency after filtering the leading N modes, per row."""
    c = np.atleast_2d(x) @ Vh.T
    return np.array([np.sqrt(np.sum(c[:, N:] ** 2, axis=1) / n_f)
                     for N in n_modes])                    # (N_SHOW+1, n_row)


def filtered(x, N):
    """The part of a single spectrum left after projecting out N modes."""
    return (x @ Vh.T)[N:] @ Vh[N:]


sys_resid = filt_rms(dT.reshape(-1, n_f)).max(axis=1)      # worst axis/LST
t21_resid = filt_rms(T21)                                  # (N_SHOW+1, n_model)
t21_pct = np.percentile(t21_resid, [5, 50, 95], axis=1)    # (3, N_SHOW+1)

In [ ]:
C_FG, C_SYS = "k", "#d55e00"
CLASS_C = ["#cc79a7", "#009e73", "#0072b2"]                 # destroyed -> survives

fig, ax = plt.subplot_mosaic(
    [["a1", "b"], ["a2", "b"]],
    figsize=(7.3, 3.2), layout="constrained",
    gridspec_kw=dict(width_ratios=[1, 1.2]),
)

for i, w in zip(show_idx, show_w):                          # (a1) in, (a2) out
    c = CLASS_C[cls[i]]
    ax["a1"].plot(freqs, T21[i] * 1e3, color=c, lw=1.1, label=f"{w:.0f} MHz wide")
    ax["a2"].plot(freqs, filtered(T21[i], N_ANCHOR) * 1e3, color=c, lw=1.1)

for key, lab, ylab in (("a1", "input", r"$T_{21}$ [mK]"),
                       ("a2", f"after filtering {N_ANCHOR} modes", "Residual [mK]")):
    ax[key].axhline(0, color="0.6", lw=0.6, ls="--", zorder=0)
    ax[key].set_ylabel(ylab, fontsize=8)
    ax[key].grid(alpha=0.2)
    ax[key].tick_params(labelsize=7)
    ax[key].text(0.03 if key == "a1" else 0.97, 0.06, lab,
                 transform=ax[key].transAxes, fontsize=7,
                 ha="left" if key == "a1" else "right", va="bottom")
ax["a1"].tick_params(labelbottom=False)
ax["a2"].set_xlabel("Frequency [MHz]", fontsize=8)
ax["a1"].legend(fontsize=6, loc="lower right", framealpha=0.9, handlelength=1.4)

b = ax["b"]
rng = np.random.default_rng(0)                              # fixed draw, reproducible
sub = rng.choice(t21_resid.shape[1], size=N_CURVES, replace=False)
for k, lab in enumerate(class_labels):                      # colour by fate at N
    kk = sub[cls[sub] == k]
    b.plot(n_modes, t21_resid[:, kk], color=CLASS_C[k], lw=0.5,
           alpha=CURVE_ALPHA, zorder=0)
    b.plot([], [], color=CLASS_C[k], lw=1.2,                 # legend proxy
           label=f"{lab} retained ({(cls == k).sum()})")
for i in show_idx:                                           # the panel (a) models
    b.plot(n_modes, t21_resid[:, i], color=CLASS_C[cls[i]], lw=1.6, zorder=2)
b.plot(n_modes, fg_resid, color=C_FG, lw=1.5, label="foreground residual")
b.plot(n_modes, sys_resid, color=C_SYS, lw=1.2, ls="--",
       label="+1 m position error (worst LST)")
b.axvline(N_ANCHOR, color="0.6", lw=0.8, ls=":", zorder=0)
b.text(N_ANCHOR - 0.3, 1e0, f"$N = {N_ANCHOR}$", fontsize=7, color="0.35",
       ha="right", va="center")
b.set_yscale("log")
b.set_xlim(0, N_SHOW)
b.set_ylim(1e-5, 3e3)
b.set_xlabel("Foreground modes filtered", fontsize=8)
b.set_ylabel("RMS over band [K]", fontsize=8)
b.grid(True, which="both", ls=":", lw=0.5, alpha=0.6)
b.tick_params(labelsize=7)
b.legend(fontsize=6.5, loc="upper right", framealpha=0.92)

fig.savefig("signal_loss.pdf", bbox_inches="tight", dpi=600)

In [ ]:
frac_above = (t21_resid > fg_resid[:, None]).mean(axis=1)
print(f"{'N':>3} {'fgnd':>9} {'pos err':>9} {'21cm p50':>9} {'21cm p95':>9} "
      f"{'frac>fgnd':>10}   (mK)")
for N in (6, 8, N_ANCHOR, 12, 15):
    print(f"{N:3d} {fg_resid[N]*1e3:9.3f} {sys_resid[N]*1e3:9.3f} "
          f"{t21_pct[1, N]*1e3:9.3f} {t21_pct[2, N]*1e3:9.3f} "
          f"{frac_above[N]:10.2f}")

keep = t21_resid[N_ANCHOR] / t21_resid[0]
print(f"\nAt N = {N_ANCHOR}: median model keeps {np.median(keep)*100:.0f}% of its "
      f"RMS ({t21_pct[1, N_ANCHOR]*1e3:.2f} mK), while the foreground residual is "
      f"{fg_resid[N_ANCHOR]*1e3:.2f} mK and the worst-case +1 m position error is "
      f"{sys_resid[N_ANCHOR]*1e3:.2f} mK.")
print(f"{frac_above[N_ANCHOR]*100:.0f}% of the {t21_resid.shape[1]} models retain "
      f"more signal than the foreground residual.")

# What separates the classes: at matched depth it is trough width, not amplitude.
width = (T21 < T21.min(axis=1, keepdims=True) / 2).sum(axis=1) * (freqs[1] - freqs[0])
depth = -T21.min(axis=1) * 1e3
window = (depth > 80) & (depth < 160)
print()
for k, lab in enumerate(class_labels):
    m, mw = cls == k, (cls == k) & window
    print(f"{lab:>6s} retained: {m.sum():4d} models, median depth "
          f"{np.median(depth[m]):6.1f} mK; at matched depth (80-160 mK) "
          f"n={mw.sum():3d}, median trough width {np.median(width[mw]):3.0f} MHz")